**KHẢO SÁT DỮ LIỆU**

In [ ]:

import pandas as pd


def load_dataset(path: str, sheet=None) -> pd.DataFrame:
    """Đọc dataset từ CSV / TSV / Excel / JSON / Parquet."""
    p = path.lower()

    if p.endswith((".xlsx", ".xlsm", ".xls")):
        return pd.read_excel(
            path,
            sheet_name=sheet if sheet is not None else 0
        )
    elif p.endswith(".csv"):
        return pd.read_csv(path)
    elif p.endswith(".tsv"):
        return pd.read_csv(path, sep="\t")
    elif p.endswith(".json"):
        return pd.read_json(path)
    elif p.endswith(".parquet"):
        return pd.read_parquet(path)
    else:
        raise ValueError("Định dạng file không được hỗ trợ.")


def numeric_range(s: pd.Series) -> dict:
    """Thống kê miền giá trị của cột số."""
    clean = s.dropna()

    info = {
        "loại": "số",
        "thiếu": int(s.isna().sum())
    }

    if clean.empty:
        info["ghi chú"] = "Không có dữ liệu hợp lệ"
        return info

    info.update({
        "min": clean.min(),
        "max": clean.max(),
        "khoảng (max-min)": clean.max() - clean.min(),
        "trung bình": round(clean.mean(), 4),
        "trung vị": clean.median(),
        "độ lệch chuẩn": round(clean.std(), 4)
    })

    q1 = clean.quantile(0.25)
    q3 = clean.quantile(0.75)
    iqr = q3 - q1

    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    info["ngoại lai (IQR)"] = int(
        ((clean < low) | (clean > high)).sum()
    )
    info["ngưỡng IQR"] = f"[{low:.4g}, {high:.4g}]"

    return info


def datetime_range(s: pd.Series) -> dict:
    """Thống kê miền giá trị của cột ngày giờ."""
    clean = s.dropna()

    info = {
        "loại": "ngày giờ",
        "thiếu": int(s.isna().sum())
    }

    if clean.empty:
        info["ghi chú"] = "Không có dữ liệu hợp lệ"
        return info

    info.update({
        "min": clean.min(),
        "max": clean.max(),
        "khoảng thời gian": clean.max() - clean.min()
    })

    return info


def categorical_range(
    s: pd.Series,
    max_unique: int = 10
) -> dict:
    """Thống kê miền giá trị của cột phân loại."""
    clean = s.dropna()
    n_unique = clean.nunique()

    info = {
        "loại": "phân loại/chuỗi",
        "thiếu": int(s.isna().sum()),
        "số giá trị duy nhất": int(n_unique)
    }

    if n_unique <= max_unique:
        info["các giá trị"] = sorted(
            clean.astype(str).unique().tolist()
        )
    else:
        top = clean.value_counts().head(5)
        info["top 5"] = {
            str(k): int(v) for k, v in top.items()
        }

    return info


def check_ranges(
    df: pd.DataFrame,
    max_unique: int = 10
) -> dict:
    """Kiểm tra miền giá trị của tất cả các cột."""
    result = {}

    for col in df.columns:
        s = df[col]

        if pd.api.types.is_datetime64_any_dtype(s):
            result[col] = datetime_range(s)
        elif pd.api.types.is_bool_dtype(s):
            result[col] = categorical_range(s, max_unique)
        elif pd.api.types.is_numeric_dtype(s):
            result[col] = numeric_range(s)
        else:
            result[col] = categorical_range(s, max_unique)

    return result


def print_report(df: pd.DataFrame, result: dict) -> None:
    """In báo cáo thống kê."""
    print(f"Dataset: {df.shape[0]} dòng x {df.shape[1]} cột")
    print("=" * 60)

    for col, info in result.items():
        print(f"\n[{col}]")

        for k, v in info.items():
            print(f"  {k}: {v}")


def main():
    from google.colab import files

    print("Vui lòng tải dataset lên.")
    uploaded = files.upload()

    if not uploaded:
        print("Bạn chưa tải file lên.")
        return

    path = next(iter(uploaded))

    # Nhập tên sheet nếu là file Excel
    sheet = None
    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):
        sheet_input = input(
            "Nhập tên sheet Excel (Enter để dùng sheet đầu tiên): "
        ).strip()

        if sheet_input:
            sheet = sheet_input

    try:
        df = load_dataset(path, sheet)
        result = check_ranges(df, max_unique=10)
        print_report(df, result)

    except Exception as e:
        print(f"Lỗi khi xử lý dữ liệu: {e}")


# Chạy chương trình trong Google Colab
main()


Vui lòng tải dataset lên.


Saving cdc_brfss_2023_stroke_decoded.csv to cdc_brfss_2023_stroke_decoded (1).csv
Dataset: 190710 dòng x 18 cột

[age]
  loại: số
  thiếu: 0
  min: 18
  max: 80
  khoảng (max-min): 62
  trung bình: 49.2304
  trung vị: 50.0
  độ lệch chuẩn: 14.6391
  ngoại lai (IQR): 0
  ngưỡng IQR: [5, 93]

[sex]
  loại: phân loại/chuỗi
  thiếu: 0
  số giá trị duy nhất: 2
  các giá trị: ['Female', 'Male']

[employment]
  loại: phân loại/chuỗi
  thiếu: 0
  số giá trị duy nhất: 7
  các giá trị: ['A homemaker', 'A student', 'Employed for wages', 'Out of work for 1 year or more', 'Out of work for less than 1 year', 'Self-employed', 'Unable to work']

[bmi_category]
  loại: phân loại/chuỗi
  thiếu: 0
  số giá trị duy nhất: 4
  các giá trị: ['Normal weight', 'Obese', 'Overweight', 'Underweight']

[smoking_status]
  loại: phân loại/chuỗi
  thiếu: 0
  số giá trị duy nhất: 4
  các giá trị: ['Current smoker - every day', 'Current smoker - some days', 'Former smoker', 'Never smoked']

[heavy_drinking]
  loại: phâ

**MÔ HÌNH AdaBoost:**

In [1]:

import pandas as pd
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    FunctionTransformer
)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    classification_report
)

# 1. DOC DU LIEU

X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_val = pd.read_csv("y_val.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

# 2. KIEM TRA DU LIEU

print("=" * 70)
print("DATA CHECK")
print("=" * 70)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

if not (
    list(X_train.columns)
    == list(X_val.columns)
    == list(X_test.columns)
):
    raise ValueError(
        "Train, Validation va Test phai co cung feature va thu tu."
    )

if y_train.isna().any() or y_val.isna().any() or y_test.isna().any():
    raise ValueError("Nhan y co gia tri thieu.")

print("\nPhan bo nhan:")
print("Train:\n", y_train.value_counts())
print("Validation:\n", y_val.value_counts())
print("Test:\n", y_test.value_counts())

# 3. CHIA NHOM FEATURE

binary_features = [
    'sex',
    'heavy_drinking',
    'physical_activity',
    'high_bp',
    'high_cholesterol',
    'heart_attack',
    'heart_disease',
    'asthma',
    'kidney_disease'
]

nominal_features = [
    'employment',
    'smoking_status',
    'diabetes'
]

ordinal_features = [
    'bmi_category',
    'general_health'
]

numerical_features = [
    'age',
    'physical_health_days',
    'mental_health_days'
]

all_features = (
    binary_features
    + nominal_features
    + ordinal_features
    + numerical_features
)

missing_features = [
    col for col in all_features
    if col not in X_train.columns
]

if missing_features:
    raise ValueError(
        f"Khong tim thay feature: {missing_features}"
    )

# 4. BINARY ENCODING

def binary_encode(X):
    X = pd.DataFrame(
        X,
        columns=binary_features
    ).copy()

    mapping = {
        'Male': 0,
        'Female': 1,
        'male': 0,
        'female': 1,
        'Yes': 1,
        'No': 0,
        'yes': 1,
        'no': 0
    }

    for col in binary_features:
        X[col] = X[col].replace(mapping)
        X[col] = pd.to_numeric(
            X[col],
            errors='coerce'
        )

    return X.values


binary_pipeline = Pipeline([
    (
        'encoder',
        FunctionTransformer(
            binary_encode,
            feature_names_out='one-to-one'
        )
    ),
    (
        'imputer',
        SimpleImputer(strategy='most_frequent')
    )
])

# 5. NOMINAL ENCODING

nominal_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='most_frequent')
    ),
    (
        'onehot',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        )
    )
])

# 6. ORDINAL ENCODING

bmi_categories = [
    'Underweight',
    'Normal weight',
    'Overweight',
    'Obese'
]

health_categories = [
    'Poor',
    'Fair',
    'Good',
    'Very good',
    'Excellent'
]

ordinal_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='most_frequent')
    ),
    (
        'encoder',
        OrdinalEncoder(
            categories=[
                bmi_categories,
                health_categories
            ],
            handle_unknown='use_encoded_value',
            unknown_value=-1
        )
    )
])

# 7. NUMERICAL ENCODING

numerical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    )
])

# 8. PREPROCESSOR

preprocessor = ColumnTransformer(
    transformers=[
        (
            'binary',
            binary_pipeline,
            binary_features
        ),
        (
            'nominal',
            nominal_pipeline,
            nominal_features
        ),
        (
            'ordinal',
            ordinal_pipeline,
            ordinal_features
        ),
        (
            'numerical',
            numerical_pipeline,
            numerical_features
        )
    ],
    remainder='drop'
)

# 9. ADABOOST MODEL

base_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

ada_model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

ada_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', ada_model)
])

# 10. TRAIN

print("\n" + "=" * 70)
print("TRAINING ADABOOST")
print("=" * 70)

start_time = time.perf_counter()

ada_pipeline.fit(
    X_train,
    y_train
)

training_time = time.perf_counter() - start_time

print(f"Training time: {training_time:.4f} seconds")
print(f"Training time: {training_time / 60:.4f} minutes")

# 11. HAM DANH GIA

def evaluate_model(X, y, dataset_name):
    y_pred = ada_pipeline.predict(X)
    classes = list(ada_pipeline.classes_)

    if len(classes) != 2 or 1 not in classes:
        raise ValueError(
            "Can mo hinh nhi phan voi nhan 0 va 1; "
            "nhan Stroke phai la 1."
        )

    stroke_index = classes.index(1)

    y_proba = ada_pipeline.predict_proba(X)[:, stroke_index]

    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(
        y, y_pred, pos_label=1, zero_division=0
    )
    recall = recall_score(
        y, y_pred, pos_label=1, zero_division=0
    )
    macro_f1 = f1_score(
        y, y_pred, average='macro', zero_division=0
    )
    roc_auc = roc_auc_score(y, y_proba)
    pr_auc = average_precision_score(y, y_proba)

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Macro F1 : {macro_f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")

    cm = confusion_matrix(y, y_pred, labels=[0, 1])

    print("\nConfusion Matrix:")
    print(cm)

    tn, fp, fn, tp = cm.ravel()

    print("TN =", tn)
    print("FP =", fp)
    print("FN =", fn)
    print("TP =", tp)

    print("\nClassification Report:")
    print(
        classification_report(
            y,
            y_pred,
            labels=[0, 1],
            target_names=['No Stroke', 'Stroke'],
            zero_division=0
        )
    )

    return {
        'Dataset': dataset_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'Macro F1': macro_f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    }

# 12. DANH GIA TREN CA 3 TAP

train_results = evaluate_model(
    X_train, y_train, "TRAIN RESULTS"
)

val_results = evaluate_model(
    X_val, y_val, "VALIDATION RESULTS"
)

test_results = evaluate_model(
    X_test, y_test, "TEST RESULTS"
)

# 13. BANG TONG HOP

results = pd.DataFrame([
    train_results,
    val_results,
    test_results
])

print("\n" + "=" * 70)
print("RESULT TABLE")
print("=" * 70)

print(results.to_string(index=False))

DATA CHECK
X_train: (152568, 17)
X_val  : (19071, 17)
X_test : (19071, 17)
y_train: (152568,)
y_val  : (19071,)
y_test : (19071,)

Phan bo nhan:
Train:
 stroke
0    148172
1      4396
Name: count, dtype: int64
Validation:
 stroke
0    18521
1      550
Name: count, dtype: int64
Test:
 stroke
0    18522
1      549
Name: count, dtype: int64

TRAINING ADABOOST


/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)


Training time: 16.1928 seconds
Training time: 0.2699 minutes


/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)
/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)



TRAIN RESULTS
Accuracy : 0.9707
Precision: 0.3957
Recall   : 0.0332
Macro F1 : 0.5232
ROC-AUC  : 0.8439
PR-AUC   : 0.1744

Confusion Matrix:
[[147949    223]
 [  4250    146]]
TN = 147949
FP = 223
FN = 4250
TP = 146

Classification Report:
              precision    recall  f1-score   support

   No Stroke       0.97      1.00      0.99    148172
      Stroke       0.40      0.03      0.06      4396

    accuracy                           0.97    152568
   macro avg       0.68      0.52      0.52    152568
weighted avg       0.96      0.97      0.96    152568



/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)
/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)



VALIDATION RESULTS
Accuracy : 0.9699
Precision: 0.3065
Recall   : 0.0345
Macro F1 : 0.5234
ROC-AUC  : 0.8380
PR-AUC   : 0.1641

Confusion Matrix:
[[18478    43]
 [  531    19]]
TN = 18478
FP = 43
FN = 531
TP = 19

Classification Report:
              precision    recall  f1-score   support

   No Stroke       0.97      1.00      0.98     18521
      Stroke       0.31      0.03      0.06       550

    accuracy                           0.97     19071
   macro avg       0.64      0.52      0.52     19071
weighted avg       0.95      0.97      0.96     19071



/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)
/tmp/ipykernel_668/7284206.py:138: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[col] = X[col].replace(mapping)



TEST RESULTS
Accuracy : 0.9707
Precision: 0.4151
Recall   : 0.0401
Macro F1 : 0.5291
ROC-AUC  : 0.8267
PR-AUC   : 0.1676

Confusion Matrix:
[[18491    31]
 [  527    22]]
TN = 18491
FP = 31
FN = 527
TP = 22

Classification Report:
              precision    recall  f1-score   support

   No Stroke       0.97      1.00      0.99     18522
      Stroke       0.42      0.04      0.07       549

    accuracy                           0.97     19071
   macro avg       0.69      0.52      0.53     19071
weighted avg       0.96      0.97      0.96     19071


RESULT TABLE
           Dataset  Accuracy  Precision   Recall  Macro F1  ROC-AUC   PR-AUC
     TRAIN RESULTS  0.970682   0.395664 0.033212  0.523194 0.843912 0.174443
VALIDATION RESULTS  0.969902   0.306452 0.034545  0.523399 0.837982 0.164063
      TEST RESULTS  0.970741   0.415094 0.040073  0.529113 0.826654 0.167561


**MÔ HÌNH Decision Tree:**

In [2]:
import pandas as pd
import time
import sklearn
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, average_precision_score)

# 1. TẢI DỮ LIỆU
X_train = pd.read_csv('X_train.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
X_test = pd.read_csv('X_test.csv')
y_test = pd.read_csv('y_test.csv').values.ravel()
# Nạp 2 file val
X_val = pd.read_csv('X_val.csv')
y_val = pd.read_csv('y_val.csv').values.ravel()

print(f"Kích thước X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Kích thước X_val  : {X_val.shape} | y_val  : {y_val.shape}")
print(f"Kích thước X_test : {X_test.shape} | y_test : {y_test.shape}")

# 2. KHAI BÁO CÁC NHÓM FEATURE
binary_features = ['sex', 'heavy_drinking', 'physical_activity', 'high_bp',
                   'high_cholesterol', 'heart_attack', 'heart_disease', 'asthma', 'kidney_disease']
nominal_features = ['employment', 'smoking_status', 'diabetes']
ordinal_features = ['bmi_category', 'general_health']
numerical_features = ['age', 'physical_health_days', 'mental_health_days']

# 3. QUY ƯỚC TIỀN XỬ LÝ (PREPROCESSING)
# 3.1. Xử lý Binary (Yes=1, No=0, Male=0, Female=1)
def binary_encode(X):
    X_copy = X.copy()
    X_copy = X_copy.replace({'Yes': 1, 'No': 0, 'Male': 0, 'Female': 1}).infer_objects(copy=False)
    return X_copy

binary_transformer = FunctionTransformer(binary_encode, validate=False)

# 3.2. Xử lý Ordinal
# Định nghĩa thứ tự chính xác cho OrdinalEncoder
bmi_categories = ['Underweight', 'Normal weight', 'Overweight', 'Obese']
health_categories = ['Poor', 'Fair', 'Good', 'Very good', 'Excellent']
ordinal_transformer = OrdinalEncoder(
    categories=[bmi_categories, health_categories],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

# 3.3. Xử lý Nominal
nominal_transformer = OneHotEncoder(handle_unknown='ignore')

# 3.4. Ghép nối các bộ xử lý bằng ColumnTransformer
# Numerical giữ nguyên (passthrough) vì Decision Tree không cần Scale
preprocessor = ColumnTransformer(
    transformers=[
        ('bin', binary_transformer, binary_features),
        ('nom', nominal_transformer, nominal_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('num', 'passthrough', numerical_features)
    ])

# 4. TẠO PIPELINE VÀ HUẤN LUYỆN MÔ HÌNH
# Khởi tạo mô hình Decision Tree (dùng tham số mặc định cho baseline mất cân bằng)
dt_model = DecisionTreeClassifier(random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', dt_model)
])

# Đo thời gian huấn luyện
start_time = time.time()
pipeline.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"\nThời gian huấn luyện: {train_time:.2f} giây")

# 5. ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TRAIN
y_train_pred = pipeline.predict(X_train)
y_train_proba = pipeline.predict_proba(X_train)[:, 1]

print("\n" + "="*50)
print("       KẾT QUẢ TRÊN TẬP HUẤN LUYỆN (TRAIN)")
print("="*50)
print("Accuracy               :", round(accuracy_score(y_train, y_train_pred), 4))
print("Macro F1-score         :", round(f1_score(y_train, y_train_pred, average='macro'), 4))
print("Precision (Stroke = 1) :", round(precision_score(y_train, y_train_pred, pos_label=1, zero_division=0), 4))
print("Recall (Stroke = 1)    :", round(recall_score(y_train, y_train_pred, pos_label=1), 4))
print("F1-score (Stroke = 1)  :", round(f1_score(y_train, y_train_pred, pos_label=1), 4))
print("ROC-AUC                :", round(roc_auc_score(y_train, y_train_proba), 4))
print("PR-AUC                 :", round(average_precision_score(y_train, y_train_proba), 4))
print("\nMa trận nhầm lẫn (Train):")
print(confusion_matrix(y_train, y_train_pred))

# 6. ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST VALIDATION
y_val_pred = pipeline.predict(X_val)
y_val_proba = pipeline.predict_proba(X_val)[:, 1]

print("\n" + "="*50)
print("       KẾT QUẢ TRÊN TẬP KIỂM ĐỊNH (VALIDATION)")
print("="*50)
print("Accuracy               :", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision (Stroke = 1) :", round(precision_score(y_val, y_val_pred, pos_label=1, zero_division=0), 4))
print("Recall (Stroke = 1)    :", round(recall_score(y_val, y_val_pred, pos_label=1), 4))
print("Macro F1-score         :", round(f1_score(y_val, y_val_pred, average='macro'), 4))
print("ROC-AUC                :", round(roc_auc_score(y_val, y_val_proba), 4))
print("PR-AUC                 :", round(average_precision_score(y_val, y_val_proba), 4))
print("\nMa trận nhầm lẫn (Validation):")
print(confusion_matrix(y_val, y_val_pred))

# 7. ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST
y_test_pred = pipeline.predict(X_test)
y_test_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print("       KẾT QUẢ TRÊN TẬP KIỂM THỬ (TEST)")
print("="*50)
print("Accuracy               :", round(accuracy_score(y_test, y_test_pred), 4))
print("Precision (Stroke = 1) :", round(precision_score(y_test, y_test_pred, pos_label=1, zero_division=0), 4))
print("Recall (Stroke = 1)    :", round(recall_score(y_test, y_test_pred, pos_label=1), 4))
print("Macro F1-score         :", round(f1_score(y_test, y_test_pred, average='macro'), 4))
print("ROC-AUC                :", round(roc_auc_score(y_test, y_test_proba), 4))
print("PR-AUC                 :", round(average_precision_score(y_test, y_test_proba), 4))
print("\nMa trận nhầm lẫn (Test):")
print(confusion_matrix(y_test, y_test_pred))

# Đánh giá Overfitting
y_train_pred = pipeline.predict(X_train)
y_train_proba = pipeline.predict_proba(X_train)[:, 1]

print("\n" + "="*50)
print("       ĐỐI SÁNH OVERFITTING (TRAIN VALIDATION vs TEST)")
print("="*50)
print(f"{'Chỉ số':<22} | {'Train':<10} | {'Validation':<10} | {'Test':<10}")
print("-" * 60)
print(f"{'Accuracy':<22} | {accuracy_score(y_train, y_train_pred):<10.4f} | {accuracy_score(y_val, y_val_pred):<10.4f} | {accuracy_score(y_test, y_test_pred):<10.4f}")
print(f"{'Macro F1-score':<22} | {f1_score(y_train, y_train_pred, average='macro'):<10.4f} | {f1_score(y_val, y_val_pred, average='macro'):<10.4f} | {f1_score(y_test, y_test_pred, average='macro'):<10.4f}")
print(f"{'Recall (Stroke = 1)':<22} | {recall_score(y_train, y_train_pred, pos_label=1):<10.4f} | {recall_score(y_val, y_val_pred, pos_label=1):<10.4f} | {recall_score(y_test, y_test_pred, pos_label=1):<10.4f}")
print(f"{'F1-score (Stroke = 1)':<22} | {f1_score(y_train, y_train_pred, pos_label=1):<10.4f} | {f1_score(y_val, y_val_pred, pos_label=1):<10.4f} | {f1_score(y_test, y_test_pred, pos_label=1):<10.4f}")
print(f"{'ROC-AUC':<22} | {roc_auc_score(y_train, y_train_proba):<10.4f} | {roc_auc_score(y_val, y_val_proba):<10.4f} | {roc_auc_score(y_test, y_test_proba):<10.4f}")
print("="*50)

Kích thước X_train: (152568, 17) | y_train: (152568,)
Kích thước X_val  : (19071, 17) | y_val  : (19071,)
Kích thước X_test : (19071, 17) | y_test : (19071,)

Thời gian huấn luyện: 1.81 giây

       KẾT QUẢ TRÊN TẬP HUẤN LUYỆN (TRAIN)
Accuracy               : 0.9975
Macro F1-score         : 0.9766
Precision (Stroke = 1) : 0.9985
Recall (Stroke = 1)    : 0.9142
F1-score (Stroke = 1)  : 0.9545
ROC-AUC                : 0.9998
PR-AUC                 : 0.9929

Ma trận nhầm lẫn (Train):
[[148166      6]
 [   377   4019]]

       KẾT QUẢ TRÊN TẬP KIỂM ĐỊNH (VALIDATION)
Accuracy               : 0.9434
Precision (Stroke = 1) : 0.1236
Recall (Stroke = 1)    : 0.1582
Macro F1-score         : 0.5547
ROC-AUC                : 0.5597
PR-AUC                 : 0.0444

Ma trận nhầm lẫn (Validation):
[[17904   617]
 [  463    87]]

       KẾT QUẢ TRÊN TẬP KIỂM THỬ (TEST)
Accuracy               : 0.9432
Precision (Stroke = 1) : 0.093
Recall (Stroke = 1)    : 0.1111
Macro F1-score         : 0.536
ROC-AUC  

**MÔ HÌNH Stacking:**

In [6]:
import time
import platform
import os

import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    FunctionTransformer
)
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv").squeeze("columns")
y_val = pd.read_csv("y_val.csv").squeeze("columns")
y_test = pd.read_csv("y_test.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation target distribution:")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))
environment_info = {
    "Python": platform.python_version(),
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "CPU logical cores": os.cpu_count(),
}

for key, value in environment_info.items():
    print(f"{key}: {value}")
binary_cols = [
    "sex",
    "heavy_drinking",
    "physical_activity",
    "high_bp",
    "high_cholesterol",
    "heart_attack",
    "heart_disease",
    "asthma",
    "kidney_disease"
]

nominal_cols = [
    "employment",
    "smoking_status",
    "diabetes"
]

ordinal_cols = [
    "bmi_category",
    "general_health"
]

numeric_cols = [
    "age",
    "physical_health_days",
    "mental_health_days"
]

all_features = binary_cols + nominal_cols + ordinal_cols + numeric_cols

assert len(all_features) == len(set(all_features))
assert set(all_features) == set(X_train.columns)

print("Feature grouping hợp lệ:", len(all_features), "features")

def encode_binary_features(X):
    X = X.copy()

    # ColumnTransformer truyền DataFrame nếu đầu vào là DataFrame.
    # Trường hợp dự phòng nếu nhận ndarray:
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=binary_cols)

    X["sex"] = X["sex"].map({
        "Male": 0,
        "Female": 1
    })

    yes_no_cols = [col for col in binary_cols if col != "sex"]

    for col in yes_no_cols:
        X[col] = X[col].map({
            "No": 0,
            "Yes": 1
        })

    return X.astype(float)
bmi_order = [
    "Underweight",
    "Normal weight",
    "Overweight",
    "Obese"
]

health_order = [
    "Poor",
    "Fair",
    "Good",
    "Very good",
    "Excellent"
]
dt_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            FunctionTransformer(
                encode_binary_features,
                validate=False,
                feature_names_out="one-to-one"
            ),
            binary_cols
        ),
        (
            "nominal",
            OneHotEncoder(handle_unknown="ignore"),
            nominal_cols
        ),
        (
            "ordinal",
            OrdinalEncoder(categories=[bmi_order, health_order]),
            ordinal_cols
        ),
        (
            "numeric",
            "passthrough",
            numeric_cols
        )
    ],
    remainder="drop"
)

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_pipeline = Pipeline(steps=[
    ("preprocessor", dt_preprocessor),
    ("classifier", dt_model)
])

print(dt_pipeline)

rf_binary_categories = [
    ["Male", "Female"],  # Male=0, Female=1
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"],
    ["No", "Yes"]
]

rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            OrdinalEncoder(
                categories=rf_binary_categories,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            binary_cols
        ),
        (
            "nominal",
            OneHotEncoder(handle_unknown="ignore"),
            nominal_cols
        ),
        (
            "ordinal",
            OrdinalEncoder(categories=[bmi_order, health_order]),
            ordinal_cols
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_cols
        )
    ],
    remainder="drop"
)

rf_classifier = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("classifier", rf_classifier)
])

print(rf_pipeline)

knn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            FunctionTransformer(
                encode_binary_features,
                validate=False,
                feature_names_out="one-to-one"
            ),
            binary_cols
        ),
        (
            "nominal",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            nominal_cols
        ),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[
                    bmi_order,
                    health_order
                ]
            ),
            ordinal_cols
        ),
        (
            "numerical",
            StandardScaler(),
            numeric_cols
        )
    ],
    remainder="drop"
)

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_pipeline = Pipeline(steps=[
    ("preprocessor", knn_preprocessor),
    ("classifier", knn_model)
])

print(knn_pipeline)

ada_binary_pipeline = Pipeline(steps=[
    (
        "binary_encoder",
        FunctionTransformer(
            encode_binary_features,
            validate=False,
            feature_names_out="one-to-one"
        )
    )
])

ada_nominal_pipeline = Pipeline(steps=[
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

ada_ordinal_pipeline = Pipeline(steps=[
    (
        "ordinal",
        OrdinalEncoder(
            categories=[
                bmi_order,
                health_order
            ]
        )
    )
])

ada_numerical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

ada_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            ada_binary_pipeline,
            binary_cols
        ),
        (
            "nominal",
            ada_nominal_pipeline,
            nominal_cols
        ),
        (
            "ordinal",
            ada_ordinal_pipeline,
            ordinal_cols
        ),
        (
            "numerical",
            ada_numerical_pipeline,
            numeric_cols
        )
    ],
    remainder="drop"
)

base_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

ada_model = AdaBoostClassifier(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

ada_pipeline = Pipeline(steps=[
    ("preprocessor", ada_preprocessor),
    ("classifier", ada_model)
])

print(ada_pipeline)

base_pipelines = {
    "Decision Tree": dt_pipeline,
    "Random Forest": rf_pipeline,
    "KNN": knn_pipeline,
    "AdaBoost": ada_pipeline
}

for name, pipe in base_pipelines.items():
    print(f"{name}: OK - {type(pipe).__name__}")

dt_for_stack = clone(dt_pipeline)
rf_for_stack = clone(rf_pipeline)
knn_for_stack = clone(knn_pipeline)
ada_for_stack = clone(ada_pipeline)

meta_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=42
)

print(meta_model)

estimators = [
    ("decision_tree", dt_for_stack),
    ("random_forest", rf_for_stack),
    ("knn", knn_for_stack),
    ("adaboost", ada_for_stack)
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_model,
    cv=5,
    stack_method="auto",
    passthrough=False,
    n_jobs=-1
)

print(stacking_model)

print("===== DECISION TREE =====")
print(dt_model.get_params())

print("\n===== RANDOM FOREST =====")
print(rf_classifier.get_params())

print("\n===== KNN =====")
print(knn_model.get_params())

print("\n===== ADABOOST =====")
print(ada_model.get_params())

print("\n===== META MODEL =====")
print(meta_model.get_params())

print("\n===== STACKING =====")
print({
    "cv": stacking_model.cv,
    "stack_method": stacking_model.stack_method,
    "passthrough": stacking_model.passthrough,
    "n_jobs": stacking_model.n_jobs
})

start_train = time.perf_counter()

stacking_model.fit(X_train, y_train)

training_time = time.perf_counter() - start_train

print(f"Training time: {training_time:.2f} seconds")
print(f"Training time: {training_time / 60:.2f} minutes")

start_predict = time.perf_counter()

y_pred = stacking_model.predict(X_test)
y_prob = stacking_model.predict_proba(X_test)[:, 1]

prediction_time = time.perf_counter() - start_predict

print(f"Prediction time: {prediction_time:.2f} seconds")

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"Macro F1 : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Stroke", "Stroke"],
        digits=4,
        zero_division=0
    )
)

results = pd.DataFrame({
    "Model": ["Stacking"],
    "Accuracy": [accuracy],
    "Precision_Stroke": [precision],
    "Recall_Stroke": [recall],
    "F1_Macro": [f1],
    "ROC_AUC": [roc_auc],
    "PR_AUC": [pr_auc],
    "Training_Time_Seconds": [training_time],
    "Prediction_Time_Seconds": [prediction_time],
    "CV_Folds": [5]
})

display(results)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# =========================
# PREDICT TRAIN
# =========================

y_train_pred = stacking_model.predict(X_train)
y_train_proba = stacking_model.predict_proba(X_train)[:, 1]

# =========================
# PREDICT VALIDATION
# =========================

y_val_pred = stacking_model.predict(X_val)
y_val_proba = stacking_model.predict_proba(X_val)[:, 1]

# =========================
# PREDICT TEST
# =========================

y_test_pred = stacking_model.predict(X_test)
y_test_proba = stacking_model.predict_proba(X_test)[:, 1]


# =========================
# HÀM TÍNH METRIC
# =========================

def evaluate_model(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred,
            pos_label=1,
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true,
            y_proba
        ),
        "PR-AUC": average_precision_score(
            y_true,
            y_proba
        )
    }


train_metrics = evaluate_model(
    y_train,
    y_train_pred,
    y_train_proba
)

val_metrics = evaluate_model(
    y_val,
    y_val_pred,
    y_val_proba
)

test_metrics = evaluate_model(
    y_test,
    y_test_pred,
    y_test_proba
)


# =========================
# IN KẾT QUẢ
# =========================

print("===== TRAIN =====")
for metric, value in train_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

print("\n===== VALIDATION =====")
for metric, value in val_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

print("\n===== TEST =====")
for metric, value in test_metrics.items():
    print(f"{metric:10s}: {value:.4f}")

X_train: (152568, 17)
X_val  : (19071, 17)
X_test : (19071, 17)
y_train: (152568,)
y_val  : (19071,)
y_test : (19071,)

Train target distribution:
stroke
0    148172
1      4396
Name: count, dtype: int64
stroke
0    0.971187
1    0.028813
Name: proportion, dtype: float64

Validation target distribution:
stroke
0    18521
1      550
Name: count, dtype: int64
stroke
0    0.97116
1    0.02884
Name: proportion, dtype: float64

Test target distribution:
stroke
0    18522
1      549
Name: count, dtype: int64
stroke
0    0.971213
1    0.028787
Name: proportion, dtype: float64
Python: 3.13.16
scikit-learn: 1.6.1
pandas: 2.2.3
NumPy: 2.1.3
CPU logical cores: 2
Feature grouping hợp lệ: 17 features
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('binary',
                                                  FunctionTransformer(feature_names_out='one-to-one',
                                                                      func=<function encode_binary_features 

/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training time: 296.11 seconds
Training time: 4.94 minutes
Prediction time: 41.26 seconds
Accuracy : 0.9707
Precision: 0.4082
Recall   : 0.0364
Macro F1 : 0.5260
ROC-AUC  : 0.8270
PR-AUC   : 0.1693
Confusion Matrix:
[[18493    29]
 [  529    20]]

TN: 18493
FP: 29
FN: 529
TP: 20
              precision    recall  f1-score   support

   No Stroke     0.9722    0.9984    0.9851     18522
      Stroke     0.4082    0.0364    0.0669       549

    accuracy                         0.9707     19071
   macro avg     0.6902    0.5174    0.5260     19071
weighted avg     0.9560    0.9707    0.9587     19071



,Model,Accuracy,Precision_Stroke,Recall_Stroke,F1_Macro,ROC_AUC,PR_AUC,Training_Time_Seconds,Prediction_Time_Seconds,CV_Folds
0,Stacking,0.970741,0.408163,0.03643,0.526014,0.826999,0.169282,296.110213,41.263506,5


===== TRAIN =====
Accuracy  : 0.9720
Precision : 0.6933
Recall    : 0.0514
Macro F1  : 0.5408
ROC-AUC   : 0.9014
PR-AUC    : 0.2929

===== VALIDATION =====
Accuracy  : 0.9701
Precision : 0.3077
Recall    : 0.0291
Macro F1  : 0.5190
ROC-AUC   : 0.8383
PR-AUC    : 0.1687

===== TEST =====
Accuracy  : 0.9707
Precision : 0.4082
Recall    : 0.0364
Macro F1  : 0.5260
ROC-AUC   : 0.8270
PR-AUC    : 0.1693


**MÔ HÌNH Random Forest:**

In [4]:
import os, sys, time
import numpy as np, pandas as pd, sklearn
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix, classification_report)
from sklearn.preprocessing import OneHotEncoder

POS_LABEL = 1   # giá trị của lớp Stroke trong file y (vd 1 hoặc "Yes")

BINARY  = ["sex", "heavy_drinking", "physical_activity", "high_bp", "high_cholesterol",
           "heart_attack", "heart_disease", "asthma", "kidney_disease"]
NOMINAL = ["employment", "smoking_status", "diabetes"]
BINARY_MAP = {"Yes": 1, "No": 0, "Male": 0, "Female": 1}
ORDINAL_MAP = {
    "bmi_category":   {"Underweight": 0, "Normal weight": 1, "Overweight": 2, "Obese": 3},
    "general_health": {"Poor": 0, "Fair": 1, "Good": 2, "Very good": 3, "Excellent": 4},
}

def find_csv(prefix):
    """Tìm file CSV bắt đầu bằng prefix (không phân biệt hoa/thường).
    Colab đổi tên file upload lại thành 'X_train (1).csv' -> hàm này vẫn tìm được, lấy file mới nhất."""
    cands = [f for f in os.listdir(".") if f.lower().startswith(prefix.lower()) and f.lower().endswith(".csv")]
    if not cands:
        sys.exit(f"Không thấy file '{prefix}*.csv'. Các file hiện có: {os.listdir('.')}")
    return max(cands, key=os.path.getmtime)

paths = {k: find_csv(k) for k in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]}
print("Đang dùng:", paths)

X_tr, X_va, X_te = pd.read_csv(paths["X_train"]), pd.read_csv(paths["X_val"]), pd.read_csv(paths["X_test"])
y_tr = (pd.read_csv(paths["y_train"]).iloc[:, 0] == POS_LABEL).astype(int)
y_va = (pd.read_csv(paths["y_val"]).iloc[:, 0] == POS_LABEL).astype(int)
y_te = (pd.read_csv(paths["y_test"]).iloc[:, 0] == POS_LABEL).astype(int)
print(f"Train {X_tr.shape} | Val {X_va.shape} | Test {X_te.shape} | Tỉ lệ Stroke: train={y_tr.mean():.4f}, val={y_va.mean():.4f}, test={y_te.mean():.4f}")

#-----------------Preprocessing----------------------#
def encode_fixed(df):
    """Binary + Ordinal: quy tắc cố định theo đề (không học từ dữ liệu nên không rò rỉ)."""
    df = df.copy()
    for c in BINARY:
        df[c] = df[c].map(BINARY_MAP)
    for c, m in ORDINAL_MAP.items():
        df[c] = df[c].map(m)
    bad = [c for c in BINARY + list(ORDINAL_MAP) if df[c].isna().any()]
    if bad:
        sys.exit(f"Cột {bad} có giá trị ngoài bảng mã hóa. Kiểm tra bằng df[cột].unique()")
    return df

X_tr, X_va, X_te = encode_fixed(X_tr), encode_fixed(X_va), encode_fixed(X_te)

# One-Hot cho nominal: fit trên train, val và test chỉ transform; numerical giữ nguyên (passthrough)
ohe = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), NOMINAL)],
    remainder="passthrough", verbose_feature_names_out=False)
X_tr_p = ohe.fit_transform(X_tr)
X_va_p = ohe.transform(X_va)
X_te_p = ohe.transform(X_te)
features = ohe.get_feature_names_out()
print("Số feature sau encoding:", len(features))

#---------------------Train--------------------#
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
t0 = time.time()
rf.fit(X_tr_p, y_tr)
train_time = time.time() - t0
print(f"Thời gian train: {train_time:.2f}s")

#-----------------Evaluate--------------------------#
def evaluate(X, y):
    pred, proba = rf.predict(X), rf.predict_proba(X)[:, 1]
    return {"Accuracy": accuracy_score(y, pred),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0),
            "F1 (Stroke)": f1_score(y, pred, zero_division=0),                  # F1 của lớp Stroke (average="binary")
            "F1-macro": f1_score(y, pred, average="macro", zero_division=0),
            "ROC-AUC": roc_auc_score(y, proba),
            "PR-AUC": average_precision_score(y, proba)}, pred, proba

m_tr, _, _ = evaluate(X_tr_p, y_tr)
m_va, pred_va, proba_va = evaluate(X_va_p, y_va)
m_te, pred, proba = evaluate(X_te_p, y_te)
metrics = pd.DataFrame({"train": m_tr, "val": m_va, "test": m_te}).T.round(4)
cm = confusion_matrix(y_te, pred)
cm_va = confusion_matrix(y_va, pred_va)

print("===== METRIC =====\n", metrics)
print("\nChênh lệch train - val và train - test (lớn => nghi overfitting):\n",
      pd.DataFrame({"train-val": metrics.loc["train"] - metrics.loc["val"],
                    "train-test": metrics.loc["train"] - metrics.loc["test"]}).round(4))
print("\nConfusion Matrix (val) [hàng=thực tế, cột=dự đoán]\n",
      pd.DataFrame(cm_va, index=["Thực tế: No", "Thực tế: Stroke"], columns=["Dự đoán: No", "Dự đoán: Stroke"]))
print(classification_report(y_va, pred_va, target_names=["No Stroke", "Stroke"], zero_division=0))
print("\nConfusion Matrix (test) [hàng=thực tế, cột=dự đoán]\n",
      pd.DataFrame(cm, index=["Thực tế: No", "Thực tế: Stroke"], columns=["Dự đoán: No", "Dự đoán: Stroke"]))
print("\n", classification_report(y_te, pred, target_names=["No Stroke", "Stroke"], zero_division=0))

fi = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_}).sort_values("Importance", ascending=False)

#---------------------------Visualization----------------------------#
os.makedirs("results_rf", exist_ok=True)

Đang dùng: {'X_train': 'X_train.csv', 'y_train': 'y_train.csv', 'X_val': 'X_val.csv', 'y_val': 'y_val.csv', 'X_test': 'X_test.csv', 'y_test': 'y_test.csv'}
Train (152568, 17) | Val (19071, 17) | Test (19071, 17) | Tỉ lệ Stroke: train=0.0288, val=0.0288, test=0.0288
Số feature sau encoding: 29
Thời gian train: 17.73s
===== METRIC =====
        Accuracy  Precision  Recall  F1 (Stroke)  F1-macro  ROC-AUC  PR-AUC
train    0.9974     0.9963  0.9145       0.9536    0.9762   0.9992  0.9874
val      0.9697     0.1316  0.0091       0.0170    0.5008   0.7729  0.1137
test     0.9695     0.2000  0.0200       0.0364    0.5105   0.7766  0.1098

Chênh lệch train - val và train - test (lớn => nghi overfitting):
              train-val  train-test
Accuracy        0.0277      0.0279
Precision       0.8647      0.7963
Recall          0.9054      0.8945
F1 (Stroke)     0.9366      0.9172
F1-macro        0.4754      0.4657
ROC-AUC         0.2263      0.2226
PR-AUC          0.8737      0.8776

Confusion Mat

**MÔ HÌNH K-Nearest Neighbors:**

In [4]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

# 1. Đọc dữ liệu từ 3 tập đã chia sẵn train/val/test
X_train = pd.read_csv('X_train.csv')
X_val = pd.read_csv('X_val.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_val = pd.read_csv('y_val.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

# 2. Định nghĩa các nhóm feature
binary_cols = [
    'sex', 'heavy_drinking', 'physical_activity', 'high_bp',
    'high_cholesterol', 'heart_attack', 'heart_disease', 'asthma', 'kidney_disease'
]
nominal_cols = ['employment', 'smoking_status', 'diabetes']
ordinal_cols = ['bmi_category', 'general_health']
numerical_cols = ['age', 'physical_health_days', 'mental_health_days']

# 3. Hàm chuyển đổi cho các biến nhị phân (Yes=1, No=0, Male=0, Female=1)
def encode_binary_features(X):
    X_out = X.copy()
    for col in binary_cols:
        if col == 'sex':
            X_out[col] = X_out[col].map({'Male': 0, 'Female': 1})
        else:
            X_out[col] = X_out[col].map({'No': 0, 'Yes': 1})
    return X_out

# 4. Xây dựng ColumnTransformer để xử lý từng nhóm feature
preprocessor = ColumnTransformer(
    transformers=[
        ('binary', FunctionTransformer(encode_binary_features), binary_cols),
        ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
        ('ordinal', OrdinalEncoder(categories=[
            ['Underweight', 'Normal weight', 'Overweight', 'Obese'],
            ['Poor', 'Fair', 'Good', 'Very good', 'Excellent']
        ]), ordinal_cols),
        ('numerical', StandardScaler(), numerical_cols)
    ],
    remainder='drop'
)

# 5. Xây dựng Pipeline cho mô hình KNN (k cố định = 5)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

# 6. Huấn luyện mô hình trên tập TRAIN
pipeline.fit(X_train, y_train)

# 7. Hàm đánh giá dùng chung cho từng tập
def evaluate(name, model, X, y, show_report=True):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]

    print(f"--- Tập {name} ---")
    print(f"Accuracy       : {accuracy_score(y, y_pred):.4f}")
    print(f"Precision      : {precision_score(y, y_pred, zero_division=0):.4f}")
    print(f"Recall         : {recall_score(y, y_pred, zero_division=0):.4f}")
    print(f"F1 (lớp 1)     : {f1_score(y, y_pred, zero_division=0):.4f}")
    print(f"F1_macro_score : {f1_score(y, y_pred, average='macro'):.4f}")
    print(f"ROC-AUC        : {roc_auc_score(y, y_proba):.4f}")
    print(f"PR-AUC         : {average_precision_score(y, y_proba):.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y, y_pred))
    if show_report:
        print("Classification Report:")
        print(classification_report(y, y_pred, zero_division=0))
    print()

# 8. In kết quả trên TRAIN / VAL / TEST
print("=== ĐÁNH GIÁ MÔ HÌNH KNN (k=5) ===\n")
evaluate("TRAIN", pipeline, X_train, y_train)
evaluate("VAL",   pipeline, X_val,   y_val)
evaluate("TEST",  pipeline, X_test,  y_test)

=== ĐÁNH GIÁ MÔ HÌNH KNN (k=5) ===

--- Tập TRAIN ---
Accuracy       : 0.9722
Precision      : 0.6640
Recall         : 0.0742
F1 (lớp 1)     : 0.1334
F1_macro_score : 0.5597
ROC-AUC        : 0.9614
PR-AUC         : 0.3421
Confusion matrix:
[[148007    165]
 [  4070    326]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99    148172
           1       0.66      0.07      0.13      4396

    accuracy                           0.97    152568
   macro avg       0.82      0.54      0.56    152568
weighted avg       0.96      0.97      0.96    152568


--- Tập VAL ---
Accuracy       : 0.9697
Precision      : 0.2545
Recall         : 0.0255
F1 (lớp 1)     : 0.0463
F1_macro_score : 0.5155
ROC-AUC        : 0.6118
PR-AUC         : 0.0571
Confusion matrix:
[[18480    41]
 [  536    14]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     18521
         